# Fine level clustering and annotation for brain tissue lymphocytes validation dataset

In [164]:
suppressPackageStartupMessages({
  library(Seurat)
  library(future)
  library(tidyverse)
})

In [ ]:
options(future.globals.maxSize = 1000000 * 1024^2, hpc.ncpus = 16)
plan(multicore, workers = getOption("hpc.ncpus", 1))

In [ ]:
results.dir <- "../results/tables/external/lesion_rims/cluster_annotate"
dir.create(file.path(results.dir, "subcluster"), recursive = TRUE)

# Helper functions

In [166]:
# Modified implementation of Seurat:::RunLeiden() using leidenbase::leiden_find_partition() instead of leiden::leiden() for speed
RunLeiden <- function(object,
                      partition.type = c(
                        "RBConfigurationVertexPartition",
                        "ModularityVertexPartition",
                        "RBERVertexPartition",
                        "CPMVertexPartition",
                        "SignificanceVertexPartition",
                        "SurpriseVertexPartition"
                      ),
                      initial.membership = NULL,
                      node.sizes = NULL,
                      resolution.parameter = 1,
                      random.seed = 0,
                      n.iter = 10,
                      ...) {
  input <- if (inherits(x = object, what = "list")) {
    igraph::graph_from_adj_list(adjlist = object)
  } else if (inherits(x = object, what = c("dgCMatrix", "matrix", "Matrix"))) {
    if (inherits(x = object, what = "Graph")) {
      object <- Seurat::as.sparse(x = object)
    }
    igraph::graph_from_adjacency_matrix(adjmatrix = object, weighted = TRUE)
  } else if (inherits(x = object, what = "igraph")) {
    object
  } else {
    stop("Input object must be a list, matrix, dgCMatrix, Matrix, or igraph object.")
  }

  partition <- leidenbase::leiden_find_partition(
    igraph = input,
    partition_type = partition.type,
    initial_membership = initial.membership,
    edge_weights = NULL,
    node_sizes = node.sizes,
    resolution_parameter = resolution.parameter,
    seed = ifelse(random.seed < 1, 1, random.seed),
    num_iter = n.iter
  )
  return(partition[[1]])
}

assignInNamespace("RunLeiden", RunLeiden, ns = "Seurat")

# Load data

### Full dataset

QS object generated by running notebooks in `02_prepare_external_datasets`

In [167]:
seu <- qs::qread(file = "../data/processed/external/lesion_rims/annotated/lesion_rims_lymphocytes.qs", nthreads = getOption("hpc.ncpus", 1))

In [168]:
seu

An object of class Seurat 
20521 features across 1517 samples within 1 assay 
Active assay: RNA (20521 features, 0 variable features)
 3 layers present: counts, scale.data, data
 2 dimensional reductions calculated: pca, integrated.rna

### Cluster annotation

Cluster annotation TSV generated by running `slurm/reference_map_lesion_rims.sh`

In [ ]:
cluster.annotation <- read.table(
  file = file.path(results.dir, "reference_map.tsv"),
  sep = "\t",
  header = TRUE,
  row.names = 1,
  stringsAsFactors = FALSE
) %>%
  select(cluster_fine) %>%
  mutate(
    cluster_fine = factor(
      cluster_fine,
      levels = c(
        "Artefact",
        "B_naive_transitional", "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional_1", "DC_conventional_2", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_CD16", "Mono_CD16", "Mono_CD16_IFN",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN", "T_CD4_memory_central", "T_CD4_memory_central_IFN", "T_CD4_memory_effector",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN", "T_CD8_memory_central", "T_CD8_memory_effector",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_coarse = factor(
      case_match(
        cluster_fine,
        "Artefact" ~ "Artefact",
        c("B_naive_transitional", "B_naive") ~ "B_naive",
        "B_memory" ~ "B_memory",
        "B_plasma" ~ "B_plasma",
        "DC_AXL_SIGLEC6" ~ "DC_AXL_SIGLEC6",
        c("DC_conventional_1", "DC_conventional_2") ~ "DC_conventional",
        "DC_plasmacytoid" ~ "DC_plasmacytoid",
        "Doublet" ~ "Doublet",
        "Erythrocyte" ~ "Erythrocyte",
        "Granulocyte" ~ "Granulocyte",
        "ILC" ~ "ILC",
        c("Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_CD16") ~ "Mono_CD14",
        c("Mono_CD16", "Mono_CD16_IFN") ~ "Mono_CD16",
        "NK_CD56bright" ~ "NK_CD56bright",
        "NK_CD56dim" ~ "NK_CD56dim",
        "Platelet" ~ "Platelet",
        "Progenitor" ~ "Progenitor",
        "Proliferating" ~ "Proliferating",
        c("T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN") ~ "T_CD4_naive",
        c("T_CD4_memory_central", "T_CD4_memory_central_IFN", "T_CD4_memory_effector") ~ "T_CD4_memory",
        "T_regulatory_naive" ~ "T_regulatory_naive",
        "T_regulatory_memory" ~ "T_regulatory_memory",
        c("T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN") ~ "T_CD8_naive",
        c("T_CD8_memory_central", "T_CD8_memory_effector") ~ "T_CD8_memory",
        "T_MAIT" ~ "T_MAIT",
        "T_GD" ~ "T_GD",
        "T_DN" ~ "T_DN"
      ),
      levels = c(
        "Artefact",
        "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD16",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_memory",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_memory",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_main = factor(
      case_match(
        cluster_coarse,
        "Artefact" ~ "Artefact",
        c("B_naive", "B_memory", "B_plasma") ~ "B",
        c("DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid") ~ "DC",
        "Doublet" ~ "Doublet",
        "Erythrocyte" ~ "Erythrocyte",
        "Granulocyte" ~ "Granulocyte",
        "ILC" ~ "ILC",
        c("Mono_CD14", "Mono_CD16") ~ "Mono",
        c("NK_CD56bright", "NK_CD56dim") ~ "NK",
        "Platelet" ~ "Platelet",
        "Progenitor" ~ "Progenitor",
        "Proliferating" ~ "Proliferating",
        c("T_CD4_naive", "T_CD4_memory", "T_regulatory_naive", "T_regulatory_memory",
          "T_CD8_naive", "T_CD8_memory", "T_MAIT", "T_GD", "T_DN") ~ "T"
      ),
      levels = c(
        "Artefact", "B", "DC", "Doublet", "Erythrocyte", "Granulocyte", "ILC", "Mono", "NK", "Platelet", "Progenitor", "Proliferating", "T"
      )
    )
  )
seu <- AddMetaData(seu, metadata = cluster.annotation)
Idents(seu) <- "cluster_coarse"

## T cells

In [170]:
# Subset to T cells only
sub <- subset(seu, subset = cluster_main == "T")
Idents(sub) <- "cluster_fine"

### Subcluster T cells

In [171]:
plan(sequential)
tmp <- sub %>%
  FindNeighbors(
    reduction = "integrated.rna",
    dims = 1:50,
    k.param = 30,
    annoy.metric = "cosine",
    compute.SNN = FALSE,
    graph.name = "nn"
  ) %>%
  FindClusters(
    resolution = 1.5,
    graph.name = "nn",
    algorithm = 4, # Leiden algorithm
    method = "igraph",
    n.iter = 2
  )
tmp[["cluster_sub"]] <- case_match(
  as.character(tmp$seurat_clusters),
  "1" ~ NA,
  "2" ~ NA,
  "3" ~ NA,
  "4" ~ NA,
  "5" ~ NA,
  "6" ~ NA,
  "7" ~ NA,
  "8" ~ "Artefact",
  "9" ~ NA,
  "10" ~ NA,
  "11" ~ NA
)
sub <- AddMetaData(sub, metadata = tmp[["cluster_sub"]])

Computing nearest neighbor graph



Only one graph name supplied, storing nearest-neighbor graph only

Warning message:
"Adding a command log without an assay associated with it"


### Merge subclusters with fine and coarse cluster annotations

In [172]:
sub[[]] <- sub[[]] %>%
  mutate(
    cluster_fine = factor(
      if_else(is.na(cluster_sub), cluster_fine, cluster_sub),
      levels = c(
        "Artefact",
        "T_CD4_naive",
        "T_CD4_naive_SOX4",
        "T_CD4_naive_IFN",
        "T_CD4_memory_central",
        "T_CD4_memory_central_IFN",
        "T_CD4_memory_effector",
        "T_regulatory_naive",
        "T_regulatory_memory",
        "T_CD8_naive",
        "T_CD8_naive_SOX4",
        "T_CD8_naive_IFN",
        "T_CD8_memory_central",
        "T_CD8_memory_effector",
        "T_MAIT",
        "T_GD",
        "T_DN"
      )
    ),
    cluster_coarse = factor(
      case_match(
        cluster_fine,
        "Artefact" ~ "Artefact",
        "T_CD4_naive" ~ "T_CD4_naive",
        "T_CD4_naive_SOX4" ~ "T_CD4_naive",
        "T_CD4_naive_IFN" ~ "T_CD4_naive",
        "T_CD4_memory_central" ~ "T_CD4_memory",
        "T_CD4_memory_central_IFN" ~ "T_CD4_memory",
        "T_CD4_memory_effector" ~ "T_CD4_memory",
        "T_regulatory_naive" ~ "T_regulatory_naive",
        "T_regulatory_memory" ~ "T_regulatory_memory",
        "T_CD8_naive" ~ "T_CD8_naive",
        "T_CD8_naive_SOX4" ~ "T_CD8_naive",
        "T_CD8_naive_IFN" ~ "T_CD8_naive",
        "T_CD8_memory_central" ~ "T_CD8_memory",
        "T_CD8_memory_effector" ~ "T_CD8_memory",
        "T_MAIT" ~ "T_MAIT",
        "T_GD" ~ "T_GD",
        "T_DN" ~ "T_DN"
      ),
      levels = c(
        "Artefact",
        "T_CD4_naive",
        "T_CD4_memory",
        "T_regulatory_naive",
        "T_regulatory_memory",
        "T_CD8_naive",
        "T_CD8_memory",
        "T_MAIT",
        "T_GD",
        "T_DN"
      )
    ),
    cluster_main = factor(
      case_match(
        cluster_coarse,
        "Artefact" ~ "Artefact",
        "T_CD4_naive" ~ "T",
        "T_CD4_memory" ~ "T",
        "T_regulatory_naive" ~ "T",
        "T_regulatory_memory" ~ "T",
        "T_CD8_naive" ~ "T",
        "T_CD8_memory" ~ "T",
        "T_MAIT" ~ "T",
        "T_GD" ~ "T",
        "T_DN" ~ "T"
      ),
      levels = c("Artefact", "T")
    )
  )
Idents(sub) <- "cluster_fine"

In [ ]:
write.table(
  sub[[c("cluster_main", "cluster_coarse", "cluster_fine")]],
  file = file.path(results.dir, "subcluster/subcluster_annotation_t.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = TRUE,
  col.names = TRUE
)

# Update cluster annotation

In [ ]:
subcluster.annotation <- lapply(
  c("T"), # order is important (cells present in later annotation files will overwrite earlier ones)
  function(x) {
    read.table(
      file = glue::glue("{results.dir}/subcluster/subcluster_annotation_{tolower(x)}.tsv"),
      sep = "\t",
      header = TRUE,
      row.names = 1,
      stringsAsFactors = FALSE
    )
  }
) %>%
  Reduce(
    f = function(x, y) {
      filtered <- x[!rownames(x) %in% rownames(y), , drop = FALSE]
      result <- rbind(filtered, y)
      return(result)
    },
    x = .
  )

In [175]:
cluster.annotation[rownames(subcluster.annotation), ] <- subcluster.annotation[, colnames(cluster.annotation)]
cluster.annotation <- cluster.annotation %>%
  mutate(
    cluster_fine = factor(
      cluster_fine,
      levels = c(
        "Artefact",
        "B_naive_transitional", "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional_1", "DC_conventional_2", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD14_platelet", "Mono_CD14_IL1B", "Mono_CD14_IFN", "Mono_CD14_CD16", "Mono_CD16", "Mono_CD16_IFN",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_naive_SOX4", "T_CD4_naive_IFN", "T_CD4_memory_central", "T_CD4_memory_central_IFN", "T_CD4_memory_effector",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_naive_SOX4", "T_CD8_naive_IFN", "T_CD8_memory_central", "T_CD8_memory_effector",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_coarse = factor(
      cluster_coarse,
      levels = c(
        "Artefact",
        "B_naive", "B_memory", "B_plasma",
        "DC_AXL_SIGLEC6", "DC_conventional", "DC_plasmacytoid",
        "Doublet", "Erythrocyte", "Granulocyte", "ILC",
        "Mono_CD14", "Mono_CD16",
        "NK_CD56bright", "NK_CD56dim",
        "Platelet", "Progenitor", "Proliferating",
        "T_CD4_naive", "T_CD4_memory",
        "T_regulatory_naive", "T_regulatory_memory",
        "T_CD8_naive", "T_CD8_memory",
        "T_MAIT", "T_GD", "T_DN"
      )
    ),
    cluster_main = factor(
      cluster_main,
      levels = c(
        "Artefact", "B", "DC", "Doublet", "Erythrocyte", "Granulocyte", "ILC", "Mono", "NK", "Platelet", "Progenitor", "Proliferating", "T"
      )
    )
  )
seu <- AddMetaData(seu, metadata = cluster.annotation)
Idents(seu) <- "cluster_coarse"

# Save final cluster annotations

In [ ]:
write.table(
  cluster.annotation,
  file = file.path(results.dir, "cluster_annotation_final.tsv"),
  sep = "\t",
  quote = FALSE,
  row.names = TRUE,
  col.names = TRUE
)

# Session info

In [40]:
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /ceph/project/fuggerlab/rfarooq/.conda/envs/sandbox/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
[1] C

time zone: Europe/London
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] reticulate_1.39.0  lubridate_1.9.3    forcats_1.0.0      stringr_1.5.1     
 [5] dplyr_1.1.4        purrr_1.0.2        readr_2.1.5        tidyr_1.3.1       
 [9] tibble_3.2.1       ggplot2_3.5.1      tidyverse_2.0.0    future_1.34.0     
[13] Seurat_5.1.0       SeuratObject_5.0.2 sp_2.1-4          

loaded via a namespace (and not attached):
  [1] RColorBrewer_1.1-3     jsonlite_1.8.9         magrittr_2.0.3        
  [4] spatstat.utils_3.1-0   farver_2.1.2           vctrs_0.6.5           
  [7] ROCR_1.0-11            spatstat.explore_3.2-6 base6